# `patch_dataset` — PyTorch Dataset for S1/S2 Patches

Sliding-window `torch.utils.data.Dataset` over the aligned S1/S2/GT scenes: windowed rasterio reads (or full in-memory caching), per-channel standardization, and optional flip/rotation augmentation. Converted from `src/data/patch_dataset.py`. Depends on `splits.ipynb` for `get_scene_files`, loaded below via `load_notebook()`.

In [1]:
import json

def load_notebook(nb_path, namespace=None):
    """Dependency-free .ipynb loader: execs a notebook's code cells into a namespace so
    classes/functions defined there can be reused elsewhere - no import machinery, no
    extra packages (nbformat/nbconvert aren't needed; this just reads the .ipynb's own
    JSON). Works identically here and in real Jupyter/VSCode."""
    ns = namespace if namespace is not None else {}
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)
    for cell in nb["cells"]:
        if cell["cell_type"] == "code":
            exec(compile("".join(cell["source"]), nb_path, "exec"), ns)
    return ns

import os
REPO = r"C:\Users\prajw\OneDrive - Amrita Vishwa Vidyapeetham\Desktop\SEM5\Deep Learning\flood-segmentation-sentinel12"
_splits_ns = load_notebook(os.path.join(REPO, "src", "data", "splits.ipynb"))
get_scene_files = _splits_ns["get_scene_files"]
print("Loaded get_scene_files from splits.ipynb")


splits loaded: get_scene_files
Train: 14 scenes {'BNPI': 7, 'KNPI': 7}
  BNPI_20210223
  BNPI_20210310
  BNPI_20230208
  BNPI_20230213
  BNPI_20230519
  BNPI_20240109
  BNPI_20240203
  KNPI_20210218
  KNPI_20210223
  KNPI_20220119
  KNPI_20220213
  KNPI_20220310
  KNPI_20220325
  KNPI_20220723

Test: 9 scenes {'BNPI': 4, 'KNPI': 5}
  BNPI_20220208
  BNPI_20220325
  BNPI_20231125
  BNPI_20240129
  KNPI_20230213
  KNPI_20230603
  KNPI_20240104
  KNPI_20240129
  KNPI_20240203
Loaded get_scene_files from splits.ipynb


In [2]:
import torch
from torch.utils.data import Dataset
import numpy as np
import rasterio
from rasterio.windows import Window
from pathlib import Path
from typing import List, Dict, Tuple
from tqdm import tqdm

# Computed from training set:
S2_MEAN = np.array([0.0436, 0.0557, 0.0723, 0.0838, 0.1151, 0.1598, 0.1800, 0.1974, 0.2004, 0.2233, 0.1706], dtype=np.float32)
S2_STD  = np.array([0.0182, 0.0233, 0.0272, 0.0340, 0.0380, 0.0598, 0.0686, 0.0751, 0.0728, 0.0674, 0.0606], dtype=np.float32)

S1_MEAN = np.array([-21.6081, -13.6686, -8.0403], dtype=np.float32)
S1_STD  = np.array([3.4706, 3.0644, 2.5309], dtype=np.float32)

class FloodPatchDataset(Dataset):
    def __init__(
        self,
        dataset_root: str,
        split: str,
        patch_size: int = 64,
        stride: int = 64,
        augment: bool = False,
        cache_in_memory: bool = False,
        load_s1: bool = True
    ):
        """
        PyTorch Dataset for Sentinel-1/2 Flood Segmentation using windowed reads or in-memory cache.
        """
        self.dataset_root = dataset_root
        self.split = split
        self.patch_size = patch_size
        self.stride = stride
        self.augment = augment
        self.cache_in_memory = cache_in_memory
        self.load_s1 = load_s1

        # Get aligned scene files
        self.scenes = get_scene_files(dataset_root, split)

        self.s2_readers = []
        self.s1_readers = []
        self.gt_readers = []

        self.s2_cache = {}
        self.s1_cache = {}
        self.gt_cache = {}

        self.patches = []

        print(f"Initializing {split} dataset (cache_in_memory={cache_in_memory}, load_s1={load_s1})...")

        for scene_idx, scene in enumerate(self.scenes):
            with rasterio.open(scene["S2"]) as s2_src:
                H, W = s2_src.height, s2_src.width

            if cache_in_memory:
                with rasterio.open(scene["S2"]) as s2_src:
                    s2_img = s2_src.read().astype(np.float32)
                    s2_img /= 10000.0
                    self.s2_cache[scene_idx] = s2_img
                if load_s1:
                    with rasterio.open(scene["S1"]) as s1_src:
                        self.s1_cache[scene_idx] = s1_src.read().astype(np.float32)
                with rasterio.open(scene["GT_0_8"]) as gt_src:
                    self.gt_cache[scene_idx] = gt_src.read(1).astype(np.int64)
            else:
                self.s2_readers.append(rasterio.open(scene["S2"]))
                if load_s1:
                    self.s1_readers.append(rasterio.open(scene["S1"]))
                self.gt_readers.append(rasterio.open(scene["GT_0_8"]))

            for y in range(0, H - patch_size + 1, stride):
                for x in range(0, W - patch_size + 1, stride):
                    self.patches.append({"scene_idx": scene_idx, "y": y, "x": x})

        print(f"Loaded {split} dataset with {len(self.patches)} total patches from {len(self.scenes)} scenes.")

    def close(self):
        for r in self.s2_readers:
            try: r.close()
            except: pass
        for r in self.s1_readers:
            try: r.close()
            except: pass
        for r in self.gt_readers:
            try: r.close()
            except: pass

    def __del__(self):
        self.close()

    def __len__(self) -> int:
        return len(self.patches)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        patch_info = self.patches[idx]
        scene_idx = patch_info["scene_idx"]
        y, x = patch_info["y"], patch_info["x"]

        if self.cache_in_memory:
            s2_data = self.s2_cache[scene_idx][:, y:y+self.patch_size, x:x+self.patch_size].copy()
            if self.load_s1:
                s1_data = self.s1_cache[scene_idx][:, y:y+self.patch_size, x:x+self.patch_size].copy()
            else:
                s1_data = np.zeros((3, self.patch_size, self.patch_size), dtype=np.float32)
            gt_data = self.gt_cache[scene_idx][y:y+self.patch_size, x:x+self.patch_size].copy()
        else:
            window = Window(x, y, self.patch_size, self.patch_size)
            s2_data = self.s2_readers[scene_idx].read(window=window).astype(np.float32)
            s2_data /= 10000.0

            if self.load_s1:
                s1_data = self.s1_readers[scene_idx].read(window=window).astype(np.float32)
            else:
                s1_data = np.zeros((3, self.patch_size, self.patch_size), dtype=np.float32)

            gt_data = self.gt_readers[scene_idx].read(1, window=window).astype(np.int64)

        for c in range(11):
            s2_data[c] = (s2_data[c] - S2_MEAN[c]) / S2_STD[c]

        if self.load_s1:
            for c in range(3):
                s1_data[c] = (s1_data[c] - S1_MEAN[c]) / S1_STD[c]

        if self.augment:
            if np.random.rand() > 0.5:
                s2_data = np.flip(s2_data, axis=2).copy()
                if self.load_s1:
                    s1_data = np.flip(s1_data, axis=2).copy()
                gt_data = np.flip(gt_data, axis=1).copy()

            if np.random.rand() > 0.5:
                s2_data = np.flip(s2_data, axis=1).copy()
                if self.load_s1:
                    s1_data = np.flip(s1_data, axis=1).copy()
                gt_data = np.flip(gt_data, axis=0).copy()

            rot_k = np.random.randint(0, 4)
            if rot_k > 0:
                s2_data = np.rot90(s2_data, k=rot_k, axes=(1, 2)).copy()
                if self.load_s1:
                    s1_data = np.rot90(s1_data, k=rot_k, axes=(1, 2)).copy()
                gt_data = np.rot90(gt_data, k=rot_k, axes=(0, 1)).copy()

        s2_tensor = torch.from_numpy(s2_data)
        s1_tensor = torch.from_numpy(s1_data)
        gt_tensor = torch.from_numpy(gt_data)

        return s2_tensor, s1_tensor, gt_tensor

print("patch_dataset loaded: FloodPatchDataset, S2_MEAN, S2_STD, S1_MEAN, S1_STD")


patch_dataset loaded: FloodPatchDataset, S2_MEAN, S2_STD, S1_MEAN, S1_STD


## Demo / sanity check
(equivalent to the old `tests/test_dataset.py` unit test — folded in here as a runnable check instead of a separate pytest file.)

In [3]:
import yaml, os
REPO = r"C:\Users\prajw\OneDrive - Amrita Vishwa Vidyapeetham\Desktop\SEM5\Deep Learning\flood-segmentation-sentinel12"
with open(os.path.join(REPO, "configs", "data.yaml")) as f:
    _cfg = yaml.safe_load(f)

_demo_ds = FloodPatchDataset(
    dataset_root=_cfg["dataset_root"],
    split="Test",
    patch_size=_cfg["patch_size"],
    stride=1024,  # large stride here purely to keep this demo instantiation fast
    augment=True,
)
print(f"Dataset length: {len(_demo_ds)}")

s2, s1, gt = _demo_ds[0]
print(f"S2 tensor shape: {s2.shape}, dtype: {s2.dtype}")
print(f"S1 tensor shape: {s1.shape}, dtype: {s1.dtype}")
print(f"GT tensor shape: {gt.shape}, dtype: {gt.dtype}")

assert s2.shape == (11, 64, 64)
assert s1.shape == (3, 64, 64)
assert gt.shape == (64, 64)
print("Dataset sanity check passed.")


Initializing Test dataset (cache_in_memory=False, load_s1=True)...
C:\Python314\Lib\site-packages\rasterio\__init__.py:379: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
Loaded Test dataset with 171 total patches from 9 scenes.
Dataset length: 171
S2 tensor shape: torch.Size([11, 64, 64]), dtype: torch.float32
S1 tensor shape: torch.Size([3, 64, 64]), dtype: torch.float32
GT tensor shape: torch.Size([64, 64]), dtype: torch.int64
Dataset sanity check passed.
